# Age Estimation - Google Colab Setup

Ce notebook configure l'environnement sur Google Colab pour le projet d'estimation d'âge.

## 1. Clone le repository GitHub

In [2]:
!git clone https://github.com/Lduvignacq/Deep-learning-project.git
%cd Deep-learning-project

fatal: destination path 'Deep-learning-project' already exists and is not an empty directory.
/content/Deep-learning-project
/content/Deep-learning-project


## 2. Installe les dépendances

**Note importante :** Les packages sont installés dans un ordre spécifique :
1. Packages de base et dépendances
2. PyTorch et TensorBoard
3. **albumentations** (au lieu de imgaug, compatible avec NumPy 2.0+)

✅ **Changement :** On utilise `albumentations` au lieu de `imgaug` car ce dernier n'est pas compatible avec NumPy 2.0 et Python 3.12.

In [3]:
# Mise à jour de pip et installation des dépendances
!pip install --upgrade pip -q

# Installation de better-exceptions en premier (requis par train.py)
!pip install -q better-exceptions

# Installation des packages de base
!pip install -q future pandas tqdm yacs pretrainedmodels

# Installation de PyTorch et TensorBoard
!pip install -q torch torchvision tensorboard

# Installation des packages d'images (avec albumentations au lieu de imgaug)
!pip install -q scikit-image imageio Pillow matplotlib Shapely scipy six numpy opencv-python

# Installation d'albumentations (alternative moderne à imgaug, compatible NumPy 2.0)
!pip install -q albumentations

print("✅ Installation terminée !")
print("✅ albumentations installé (compatible NumPy 2.0+)")

✅ Installation terminée !
✅ albumentations installé (compatible NumPy 2.0+)


✅ **Modification du code :** Le fichier `dataset.py` a été modifié pour utiliser `albumentations` au lieu de `imgaug`, ce qui résout tous les problèmes de compatibilité avec NumPy 2.0 et Python 3.12.

## 3. Télécharge le dataset APPA-REAL

In [ ]:
# Monte Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copie le dataset depuis Google Drive vers l'environnement Colab
# INSTRUCTIONS:
# 1. Uploadez d'abord appa-real-release.zip dans votre Google Drive (dans "Mon Drive")
# 2. Puis exécutez cette cellule

import os
zip_path = '/content/drive/MyDrive/appa-real-release.zip'

if os.path.exists(zip_path):
    print("📦 Copie et décompression du dataset depuis Google Drive...")
    !cp /content/drive/MyDrive/appa-real-release.zip .
    # Utilise -o pour overwrite automatiquement (évite les prompts)
    !unzip -o -q appa-real-release.zip
    # Supprime les fichiers macOS cachés
    !rm -rf __MACOSX
    print("✅ Dataset prêt !")
    !ls -la appa-real-release/
else:
    print(f"❌ Fichier non trouvé: {zip_path}")
    print("📁 Veuillez uploader appa-real-release.zip dans 'Mon Drive' sur Google Drive d'abord.")
    print("   Ou ajustez le chemin ci-dessus si le fichier est dans un autre dossier.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📦 Copie et décompression du dataset depuis Google Drive...


## 4. Vérifie que le dataset fonctionne

In [ ]:
# Vérification du répertoire courant et des fichiers
!pwd
!ls -la

# Test du dataset
!python dataset.py --data_dir appa-real-release

/content/Deep-learning-project/Deep-learning-project/Deep-learning-project
total 873552
drwxr-xr-x  6 root root      4096 Feb 24 14:16 .
drwxr-xr-x 10 root root      4096 Feb 24 14:15 ..
drwx------  5 root root      4096 Feb 24 13:12 appa-real-release
-rw-------  1 root root 894405507 Feb 24 14:16 appa-real-release.zip
-rw-r--r--  1 root root     19565 Feb 24 14:15 colab_setup.ipynb
-rw-r--r--  1 root root      3177 Feb 24 14:15 dataset.py
-rw-r--r--  1 root root       537 Feb 24 14:15 defaults.py
-rw-r--r--  1 root root      6296 Feb 24 14:15 demo.py
drwxr-xr-x  8 root root      4096 Feb 24 14:15 .git
-rw-r--r--  1 root root       376 Feb 24 14:15 .gitignore
-rw-r--r--  1 root root      1307 Feb 24 14:15 ignore_list.csv
drwxr-xr-x  3 root root      4096 Feb 24 14:16 __MACOSX
drwxr-xr-x  2 root root      4096 Feb 24 14:15 misc
-rw-r--r--  1 root root       500 Feb 24 14:15 model.py
-rw-r--r--  1 root root      4487 Feb 24 14:15 README.md
-rw-r--r--  1 root root       212 Feb 24 14:15 r

## 5. Vérifie le GPU disponible

In [ ]:
import torch
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Mémoire GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

CUDA disponible: True
GPU: Tesla T4
Mémoire GPU: 15.64 GB


## 6. Entraîne le modèle

⚠️ **Optimisations pour accélérer l'entraînement :**
- `TRAIN.BATCH_SIZE 64` : Batch size augmenté (32 → 64) pour plus de vitesse
- `TRAIN.WORKERS 4` : Réduit les workers (8 → 4) pour éviter la surcharge
- `TRAIN.EPOCHS 30` : Réduisez si vous voulez un test rapide (par défaut: 80)

💡 **Astuce :** Si vous obtenez "Out of Memory", remettez `TRAIN.BATCH_SIZE 32`

In [ ]:
# Essayez batch_size 64 ou 96 si le GPU le permet (plus rapide)
# Si erreur OOM, revenez à 32
!python train.py --data_dir appa-real-release --tensorboard tf_log TRAIN.BATCH_SIZE 96 TRAIN.WORKERS 6

python3: can't open file '/content/train.py': [Errno 2] No such file or directory


In [ ]:
# 🚀 VERSION RAPIDE (pour test - 10 epochs seulement)
# Décommentez cette ligne et commentez celle du dessous pour un test rapide
# !python train.py --data_dir appa-real-release --tensorboard tf_log TRAIN.BATCH_SIZE 64 TRAIN.WORKERS 4 TRAIN.EPOCHS 10

## 7. Visualise TensorBoard (optionnel)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir tf_log